# gcu-condenser

**Block models, drillholes and big point clouds — in the notebook.**

The renderer is [`@gcu/condenser`](https://gentropic.org/micro), the streaming engine behind *micro*:
quantize → Morton order → shuffled-prefix LOD → eye-dome lighting. Columns go in, and a few million
elements come up progressively — no decimation pass, no mesh conversion, no kernel round trip while
you navigate.

Run this top to bottom; everything is generated, no data files needed.

```
uv venv .venv && uv pip install --python .venv -e ".[dev]"
```

In [ ]:
import numpy as np
import gcu_condenser as cd

cd.__version__

## 1. A block model

We'll make a small deposit: a gaussian grade shell in a 60 × 60 × 30 lattice of 10 m blocks.
Everything below works the same from a DataFrame — `cd.blocks(df, x="XC", y="YC", z="ZC", value="FE")`.

In [ ]:
NX, NY, NZ = 60, 60, 30
i, j, k = np.meshgrid(np.arange(NX), np.arange(NY), np.arange(NZ), indexing="ij")
i, j, k = i.ravel(), j.ravel(), k.ravel()

x, y, z = i * 10.0 + 5, j * 10.0 + 5, k * 10.0 + 5
rng = np.random.default_rng(5)
d = np.sqrt(((i - NX/2) / 17)**2 + ((j - NY/2) / 13)**2 + ((k - NZ*0.45) / 8)**2)
fe = np.clip(58 * np.exp(-d * d * 1.1) + 6 + rng.normal(0, 1.0, d.size), 0, None)

print(f"{x.size:,} blocks")

In [ ]:
model = cd.blocks(x, y, z, value=fe, name="model", ramp="turbo")
model

Drag to orbit, right-drag (or shift-drag) to pan, wheel to zoom.

It's a **solid box** — from outside, all you see is waste. That's not a rendering problem, it's what a
block model is. To see the ore body you need a cutoff.

## 2. `threshold` — the grade shell

The mask is rebuilt in the browser from the value column that is already there, so changing a cutoff
**never re-sends data**. Re-run this cell with different numbers and watch the view above update.

In [ ]:
model.threshold = [28, 99]     # show only blocks at or above 28% Fe
model

Every style knob is live — assign and the view updates in place:

In [ ]:
model.ramp = "magma"
model.opacity = 1.0
model.block_edges = False
model.clip = [20, 60]          # clamp the colour scale to the interesting range

## 3. Drillholes

The usual three tables. The desurvey runs in the browser through **@gcu/drillhole** — the same
minimum-curvature code *micro* uses — so a hole lands in exactly the same place in both.

In [ ]:
cid, ccx, ccy, ccz = [], [], [], []
sid, sdep, saz, sdip = [], [], [], []
iid, ifrom, ito, iau = [], [], [], []

for h in range(9):
    hid = f"DH{h:02d}"
    cid.append(hid); ccx.append(120.0 + (h % 3) * 90); ccy.append(140.0 + (h // 3) * 90); ccz.append(320.0)
    for depth in (0, 60, 120, 180):                      # a slightly deviating hole
        sid.append(hid); sdep.append(float(depth)); saz.append(45.0 + h * 8); sdip.append(-72.0 + depth * 0.02)
    for f in range(0, 180, 3):                           # 3 m assay intervals
        iid.append(hid); ifrom.append(float(f)); ito.append(float(f + 3))
        iau.append(float(max(0, 4 * np.exp(-((f - 100) / 40)**2) + rng.random() * 0.6)))

collar = {"BHID": cid, "X": ccx, "Y": ccy, "Z": ccz}
survey = {"BHID": sid, "DEPTH": sdep, "AZ": saz, "DIP": sdip}
assay  = {"BHID": iid, "FROM": ifrom, "TO": ito, "AU": iau}

holes = cd.drillholes(collar, survey, assay, value="AU", name="holes", ramp="fire", radius=3.5)
holes

## 4. Stacking layers

`cd.view(...)` puts layers in **one shared frame**, so they co-register — and a local origin is also
what keeps mine-grid coordinates off the float32 wall on the GPU.

In [ ]:
n = 90_000
px, py = rng.random(n) * 600, rng.random(n) * 600
pz = 330 + np.sin(px / 70) * 14 + np.cos(py / 90) * 11 + rng.normal(0, 0.8, n)

topo = cd.points(px, py, pz, value=pz, name="topo", ramp="greys", sectioned=False)

w = cd.view(model, holes, topo, height=520)
w

Layers are addressable by name, and each keeps its own style:

In [ ]:
w["topo"].visible = False       # get the topography out of the way
w["holes"].radius = 4.0
print(w, "|", [l.name for l in w.layers])

## 5. Sections

`cut()` slices every layer that has not opted out. `topo` was built with `sectioned=False`, so it stays
whole for context while the model is cut — the usual way you look at a section in practice.

In [ ]:
w["topo"].visible = True
w.cut(axis="y", position=300, thickness=60)

In [ ]:
w.cut()                         # no arguments clears the section

## 6. Clicking round-trips into your table

The pick uses the engine's ID buffer, and a record index **is** the row you passed. Click a block or an
interval in the view above, then run this cell.

In [ ]:
sel = w.selection
print("selection:", sel)

if sel.get("name") == "model":
    r = sel["row"]
    print(f"block {r}: x={x[r]:.0f} y={y[r]:.0f} z={z[r]:.0f} fe={fe[r]:.1f}")
elif sel.get("name") == "holes":
    r = sel["row"]
    print(f"interval {r}: {iid[r]} {ifrom[r]:.0f}-{ito[r]:.0f} m, AU {iau[r]:.2f}")
else:
    print("click something in the view above, then re-run this cell")

With a DataFrame the round trip is just `df.iloc[w.selected_row]`.

## 7. Sub-blocked models

Pass `size=(dx, dy, dz)` and every block renders at its **true** dimensions — each distinct size becomes
a palette entry. Here 20 m parents with a 10 m core.

In [ ]:
xs, ys, zs, dx, dy, dz, val = [], [], [], [], [], [], []
for k2 in range(4):
    for j2 in range(8):
        for i2 in range(8):
            if 2 <= i2 < 6 and 2 <= j2 < 6:                       # refined core: 8 children
                for a in range(2):
                    for b in range(2):
                        for c in range(2):
                            xs.append(i2*20+5+a*10); ys.append(j2*20+5+b*10); zs.append(k2*20+5+c*10)
                            dx.append(10.0); dy.append(10.0); dz.append(10.0); val.append(30.0+i2+j2)
            else:
                xs.append(i2*20+10); ys.append(j2*20+10); zs.append(k2*20+10)
                dx.append(20.0); dy.append(20.0); dz.append(20.0); val.append(5.0+i2)

sub = cd.blocks(np.array(xs, float), np.array(ys, float), np.array(zs, float),
                value=np.array(val), size=(np.array(dx), np.array(dy), np.array(dz)),
                name="sub-blocked", ramp="viridis", block_edges=True)
print(f"{sub.count} blocks, {len(sub._extra['dim_palette'])} distinct sizes")
sub

## 8. Scale

Progressive prefix-LOD is the default path, so this is not a special mode — it is just how it draws.
(~2M blocks is about 62 MB over the notebook comm channel; that is the practical ceiling per layer.)

In [ ]:
NX2, NY2, NZ2 = 128, 128, 100
i2, j2, k2 = np.meshgrid(np.arange(NX2), np.arange(NY2), np.arange(NZ2), indexing="ij")
i2, j2, k2 = i2.ravel(), j2.ravel(), k2.ravel()
d2 = np.sqrt(((i2 - 64) / 34)**2 + ((j2 - 64) / 27)**2 + ((k2 - 45) / 22)**2)
big_fe = np.clip(60 * np.exp(-d2 * d2) + 5, 0, None)
print(f"{i2.size:,} blocks")

cd.blocks(i2 * 10.0, j2 * 10.0, k2 * 10.0, value=big_fe, ramp="turbo", threshold=[30, 99], height=520)

## The knobs

Per **layer** (`model.ramp = ...`, or `w["holes"].radius = ...`):

| trait | |
|---|---|
| `color` | `'z'` · `'value'` · `'category'` · `'rgb'` · `'flat'` |
| `ramp` | `viridis` · `magma` · `turbo` · `greys` · `spectral` · `fire` |
| `clip` | `[lo, hi]` — clamp the colour scale |
| `threshold` | `[lo, hi]` — cutoff on the value column |
| `filter_mode` | `'isolate'` (hide the rest) or `'dim'` |
| `opacity` | screen-door see-through |
| `visible`, `point_size`, `as_points`, `block_edges`, `radius` | |
| `sectioned` | `True` · `False` (exempt) · `'front'` · `'behind'` |
| `selected` | read back: the row picked on this layer |

Per **view** (`w.height = ...`):
`section` (or `w.cut(...)`), `background`, `height`, `edl`, `edl_strength`, `budget`, `selection`, `w.fit()`.

---

Two honest notes: `filter_mode='dim'` suits **point clouds** — on a solid block model the dimmed blocks
still occlude, so use `'isolate'` there. And `opacity` is a screen-door dither (real depth, no sorting),
which sees a few blocks deep, not through a whole model.